# Nepal, 26 August 2026, in 3D

A 3D story map of what the 26 August 2026 flood did on the Bhote Koshi and Trishuli, and what is still not known, built with the JupyterGIS Python API. Run all the cells, then open `nepal-flood-3d.jGIS` and start the story.

Each segment sets a camera tilt (`pitch`) and rotation (`bearing`) over 3D terrain, so the story uses the MapLibre renderer.

The text, figures and data come from [arjxn-py/nepal-flood-26-aug](https://github.com/arjxn-py/nepal-flood-26-aug), which lists every source and its licence.

In [ ]:
from jupytergis import GISDocument, constant, field, when

doc = GISDocument(
    "nepal-flood-3d.jGIS",
    map_adapter="maplibre",
    longitude=85.5204,
    latitude=28.289,
    zoom=11.0,
    pitch=45,
    bearing=30,
)
await doc.ready()

for layer in doc.layers:
    doc.remove_layer(layer)

Satellite imagery draped over 3D terrain, with place names on top of everything else.

In [ ]:
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite imagery",
    attribution="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
)
doc.add_terrain_layer(
    url="https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png",
    name="Terrain",
    exaggeration=1.3,
)

How far the water had got, worked out from the river line. Each 1 km piece of the river gets the time the water reached it: 08:37 at the slope, about 09:00 at Timure (ICIMOD's "around 9:00", read here as Timure), then the four times reported further down. Between two timed places the time is spread evenly by distance, because nobody timed those stretches. The flooded area is the one UNOSAT mapped, and each bit of it gets the time of the nearest point on the river. The 21 km past Devghat were never timed, so they stay out. The water is drawn 20 m high so it shows in 3D. That is not how deep it was; nobody measured that.

In [ ]:
from datetime import datetime, timedelta
from itertools import pairwise
from math import asin, atan2, cos, degrees, radians, sin, sqrt
from zoneinfo import ZoneInfo

import requests
from shapely import MultiPoint, set_precision, unary_union, voronoi_polygons
from shapely.geometry import mapping, shape
from shapely.ops import transform

DATA = "https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/"

bands = requests.get(DATA + "reach-bands.geojson").json()["features"]
places = requests.get(DATA + "timed-places.geojson").json()["features"]
TIMED = [
    (0, "08:37"),
    (13.7, "09:00"),
    *sorted((f["properties"]["km"], f["properties"]["arrival_local"]) for f in places),
]


def minutes(hhmm):
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m) - (8 * 60 + 37)


COLLAPSE = datetime(2026, 8, 26, 8, 37, tzinfo=ZoneInfo("Asia/Kathmandu"))


def at(minutes_after):
    return int((COLLAPSE + timedelta(minutes=minutes_after)).timestamp() * 1000)


def metres(a, b):
    lon1, lat1, lon2, lat2 = map(radians, (*a, *b))
    h = (
        sin((lat2 - lat1) / 2) ** 2
        + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    )
    return 2 * 6371008.8 * asin(sqrt(h))


def heading(a, b):
    lon1, lat1, lon2, lat2 = map(radians, (*a, *b))
    y = sin(lon2 - lon1) * cos(lat2)
    x = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(lon2 - lon1)
    return degrees(atan2(y, x))


river = []
for band in sorted(bands, key=lambda f: f["properties"]["km_start"]):
    coords = band["geometry"]["coordinates"]
    start, end = band["properties"]["km_start"], band["properties"]["km_end"]
    steps = [metres(a, b) for a, b in pairwise(coords)]
    total, run = sum(steps), 0
    river.append((start, coords[0]))
    for coord, step in zip(coords[1:], steps, strict=True):
        run += step
        river.append((start + (end - start) * run / total, coord))


def point_at(km):
    for (k1, c1), (k2, c2) in pairwise(river):
        if k1 <= km <= k2:
            t = (km - k1) / (k2 - k1) if k2 > k1 else 0
            return [
                round(c1[0] + t * (c2[0] - c1[0]), 5),
                round(c1[1] + t * (c2[1] - c1[1]), 5),
            ]
    return river[-1][1]


def minutes_at(km):
    for (k1, t1), (k2, t2) in pairwise(TIMED):
        if k1 <= km <= k2:
            return minutes(t1) + (km - k1) / (k2 - k1) * (minutes(t2) - minutes(t1))


def camera(km, looking_back=False):
    upstream, here = point_at(km - 6), point_at(km)
    angle = heading(here, upstream) if looking_back else heading(upstream, here)
    return {"center": tuple(here), "bearing": round(angle)}


wave = {"type": "FeatureCollection", "features": []}
km = 0.0
while km < TIMED[-1][0]:
    end = min(km + 1, TIMED[-1][0])
    inner = [[round(x, 5), round(y, 5)] for k, (x, y) in river if km < k < end]
    wave["features"].append(
        {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": [point_at(km), *inner, point_at(end)],
            },
            "properties": {
                "km_start": km,
                "km_end": end,
                "minutes_after_collapse": round(minutes_at(end), 1),
                "arrival": at(minutes_at(end)),
            },
        },
    )
    km = end

KX, KY = 111320 * cos(radians(27.9)), 110540


def to_metres(x, y, z=None):
    return x * KX, y * KY


def to_degrees(x, y, z=None):
    return x / KX, y / KY


unosat = requests.get(DATA + "extent-unosat.geojson").json()["features"][0]["geometry"]
unosat = transform(to_metres, shape(unosat)).buffer(0)
sites = list(
    {tuple(round(v) for v in to_metres(*c)): k for k, c in reversed(river)}.items(),
)
cells = voronoi_polygons(MultiPoint([xy for xy, _ in sites]), ordered=True).geoms
STEP = 0.1
nearest = {}
for (_, k), cell in zip(sites, cells, strict=True):
    nearest.setdefault(int(k / STEP) if k < TIMED[-1][0] else None, []).append(cell)

flood = {"type": "FeatureCollection", "features": []}
for step in sorted(s for s in nearest if s is not None):
    area = unosat.intersection(unary_union(nearest[step]))
    if not area.is_empty:
        flood["features"].append(
            {
                "type": "Feature",
                "geometry": mapping(
                    set_precision(transform(to_degrees, area.simplify(20)), 1e-5),
                ),
                "properties": {
                    "arrival": at(minutes_at(min((step + 1) * STEP, TIMED[-1][0]))),
                },
            },
        )

The data layers, read straight from the source repository. Only the channel reaches show at first; each segment turns on the layers it talks about.

In [ ]:
mapping_projects = doc.add_geojson_layer(
    path=DATA + "mapping-projects.geojson",
    name="Where the mapping projects were",
    symbology=[
        constant([0, 0, 0, 0]).encoding("fill-color"),
        constant([60, 70, 82, 0.85]).encoding("stroke-color"),
        constant(1.8).encoding("stroke-width"),
    ],
)
mapping_projects.visible = False

extent_unosat = doc.add_geojson_layer(
    path=DATA + "extent-unosat.geojson",
    name="Flood extent as UNOSAT mapped it",
    symbology=[
        constant([0, 0, 0, 0]).encoding("fill-color"),
        constant([86, 61, 140, 0.9]).encoding("stroke-color"),
        constant(2).encoding("stroke-width"),
    ],
)
extent_unosat.visible = False

extent_hot = doc.add_geojson_layer(
    path=DATA + "extent-hot.geojson",
    name="Flood extent as HOT mapped it",
    symbology=[
        constant([31, 111, 235, 0.22]).encoding("fill-color"),
        constant([31, 111, 235, 0.85]).encoding("stroke-color"),
        constant(1.6).encoding("stroke-width"),
    ],
)
extent_hot.visible = False

not_analysed = doc.add_geojson_layer(
    path=DATA + "not-analysed.geojson",
    name="Ground Copernicus could not assess",
    symbology=[
        constant([128, 128, 128, 0.28]).encoding("fill-color"),
        constant([90, 90, 90, 0.8]).encoding("stroke-color"),
        constant(1).encoding("stroke-width"),
    ],
)
not_analysed.visible = False

reaches = doc.add_geojson_layer(
    path=DATA + "reach-bands.geojson",
    name="Reaches: how steep the channel is",
    symbology=[
        when(field("band_label") == "Steep gorge (0-20 km)")
        .constant([151, 21, 12, 0.9])
        .encoding("stroke-color"),
        when(field("band_label") == "Middle valley (20-60 km)")
        .constant([220, 101, 24, 0.9])
        .encoding("stroke-color"),
        when(field("band_label") == "Open valley (60-210 km)")
        .constant([237, 209, 110, 0.9])
        .encoding("stroke-color"),
        constant(5.5).encoding("stroke-width"),
    ],
)

flood_area = doc.add_geojson_layer(
    data=flood,
    name="Where the water spread",
    symbology=[
        constant([52, 140, 220, 1]).encoding("fill-color"),
        constant(20).encoding("elevation"),
    ],
)
flood_area.visible = False

water = doc.add_geojson_layer(
    data=wave,
    name="Where the water had reached",
    symbology=[
        constant([30, 144, 255, 0.95]).encoding("stroke-color"),
        constant(4).encoding("stroke-width"),
    ],
)
water.visible = False

bridges = doc.add_geojson_layer(
    path=DATA + "bridges.geojson",
    name="Bridges, standing or destroyed",
    symbology=[
        when(field("status") == "Destroyed")
        .constant([151, 21, 12, 0.95])
        .encoding("stroke-color"),
        when(field("status") == "Standing")
        .constant([122, 134, 148, 0.8])
        .encoding("stroke-color"),
        constant(8).encoding("stroke-width"),
    ],
)
bridges.visible = False

destroyed = doc.add_geojson_layer(
    path=DATA + "destroyed-features.geojson",
    name="Marked destroyed or damaged",
    symbology=[
        constant([151, 21, 12, 0.85]).encoding("circle-fill-color"),
        constant([255, 255, 255, 1]).encoding("circle-stroke-color"),
        constant(4.6).encoding("circle-radius"),
        constant(0.8).encoding("circle-stroke-width"),
    ],
)
destroyed.visible = False

graded = doc.add_geojson_layer(
    path=DATA + "copernicus-grading.geojson",
    name="Buildings graded by Copernicus",
    symbology=[
        when(field("damage_gra") == "Destroyed")
        .constant([51, 65, 79, 0.95])
        .encoding("circle-fill-color"),
        when(field("damage_gra") == "Damaged")
        .constant([111, 131, 152, 0.95])
        .encoding("circle-fill-color"),
        when(field("damage_gra") == "Possibly damaged")
        .constant([195, 206, 217, 0.95])
        .encoding("circle-fill-color"),
        constant([255, 255, 255, 1]).encoding("circle-stroke-color"),
        constant(4.6).encoding("circle-radius"),
        constant(0.7).encoding("circle-stroke-width"),
    ],
)
graded.visible = False

schools = doc.add_geojson_layer(
    path=DATA + "schools.geojson",
    name="Schools, standing or destroyed",
    symbology=[
        when(field("status") == "Destroyed")
        .constant([151, 21, 12, 0.95])
        .encoding("circle-fill-color"),
        when(field("status") == "Standing")
        .constant([122, 134, 148, 0.8])
        .encoding("circle-fill-color"),
        constant([255, 255, 255, 1]).encoding("circle-stroke-color"),
        constant(8).encoding("circle-radius"),
        constant(1.8).encoding("circle-stroke-width"),
    ],
)
schools.visible = False

detachment = doc.add_geojson_layer(
    path=DATA + "detachment-zone.geojson",
    name="Where it came off the mountain",
    symbology=[
        constant([120, 63, 24, 0.45]).encoding("fill-color"),
        constant([120, 63, 24, 1]).encoding("stroke-color"),
        constant(2).encoding("stroke-width"),
    ],
)
detachment.visible = False

lakes = doc.add_geojson_layer(
    path=DATA + "barrier-lakes.geojson",
    name="New barrier lakes",
    symbology=[
        constant([13, 148, 156, 0.55]).encoding("fill-color"),
        constant([8, 94, 100, 1]).encoding("stroke-color"),
        constant(2).encoding("stroke-width"),
    ],
)
lakes.visible = False

timed = doc.add_geojson_layer(
    path=DATA + "timed-places.geojson",
    name="Where the wave was timed",
    symbology=[
        when(field("label") == "Galchhi 10:28")
        .constant([186, 156, 224, 0.95])
        .encoding("circle-fill-color"),
        when(field("label") == "Malekhu 11:50")
        .constant([150, 109, 203, 0.95])
        .encoding("circle-fill-color"),
        when(field("label") == "Mugling 13:00")
        .constant([113, 66, 168, 0.95])
        .encoding("circle-fill-color"),
        when(field("label") == "Devghat 15:20")
        .constant([72, 32, 120, 0.95])
        .encoding("circle-fill-color"),
        constant([255, 255, 255, 1]).encoding("circle-stroke-color"),
        constant(9).encoding("circle-radius"),
        constant(2.2).encoding("circle-stroke-width"),
    ],
)
timed.visible = False

doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}",
    name="Place names",
    attribution="Esri, HERE, Garmin, FAO, NOAA, USGS",
)

Each segment shows only the data layers it names and hides the others. Map segments also set the camera tilt and rotation; text-only segments keep the map where the last map segment left it.

In [ ]:
DATA_LAYERS = [
    mapping_projects,
    extent_unosat,
    extent_hot,
    not_analysed,
    reaches,
    bridges,
    destroyed,
    graded,
    schools,
    detachment,
    lakes,
    timed,
]


def show(*shown):
    return [{"targetLayer": layer, "visible": layer in shown} for layer in DATA_LAYERS]


def flowing(from_km, to_km, seconds):
    def animate(layer, **options):
        return {
            "targetLayer": layer,
            "visible": True,
            "animation": {
                "field": "arrival",
                "start": at(minutes_at(from_km)),
                "end": at(minutes_at(to_km)),
                "duration": seconds,
                **options,
            },
        }

    return [animate(flood_area), animate(water, clock="Asia/Kathmandu")]


GRID = 0.25
samples = [point_at(i * GRID) for i in range(int(TIMED[-1][0] / GRID) + 1)]


def smoothed(km, half=2.0):
    i, span = round(km / GRID), round(half / GRID)
    window = samples[max(i - span, 0) : max(i + span + 1, 1)]
    return [
        sum(p[0] for p in window) / len(window),
        sum(p[1] for p in window) / len(window),
    ]


def chase(km):
    return {
        "center": tuple(smoothed(km)),
        "bearing": heading(smoothed(km + 1), smoothed(km - 1)),
    }


def following(from_km, to_km, seconds):
    overrides = flowing(from_km, to_km, seconds)
    steps = round((to_km - from_km) / GRID)
    path = []
    for i in range(steps + 1):
        km = from_km + (to_km - from_km) * i / steps
        camera = chase(km)
        path.append([*camera["center"], at(minutes_at(km)), camera["bearing"]])
    overrides[-1]["animation"]["cameraPath"] = path
    return overrides


story = doc.story or doc.create_story(
    title="Nepal, 26 August 2026: what the flood did, and what is still not known",
    story_type="Vertical Scroll",
    show_gradient=False,
    markdown_segment_gap=False,
    markdown_segment_opacity=0.88,
    story_panel_opacity=0.72,
    overlay_content_width="100%",
)

In [ ]:
doc.add_story_segment(
    name="What this map is about",
    center=(85.5204, 28.289),
    zoom=11.0,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition="immediate",
    transition_time=0,
    layer_overrides=show(reaches),
    markdown="""**What the flood of 26 August 2026 did, and what is still not known.**

It started with a mountain. At **08:40** that morning about a square kilometre
broke off a slope near the China border. For a few hours it was reported as an
earthquake. It was not. It was the mountain hitting the valley floor.

It picked up water and mud on the way down, and ran **210 km** down four rivers
in one day.

**What is below**

- Where it started, and what the satellites could not see
- Three stretches of river, before and after, and why the damage fell where it did
- Who died, and where: the damage is in the mountains, the dead are on the plain
- Two new lakes that are still up there
- Why no count of this flood is final

*Scroll to begin.*
""",
)

In [ ]:
doc.add_story_segment(
    name="The ground it came off",
    center=(85.5204, 28.289),
    zoom=12.0,
    pitch=45,
    bearing=120,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.4,
    layer_overrides=show(detachment),
    markdown="""**196 hectares of mountainside are gone.**

The scar was mapped from Landsat 9 images taken on the day it happened. NDRRMA
put the mass that broke off at about a square kilometre of rock and old glacier
debris.

It sits above the Lhende Khola. That is a small river that joins the Bhote Koshi
ten kilometres below. Nothing up here could be damaged. Everything else in this
map happened downhill of it.

**On the map:** the brown outline is where it broke away
""",
)

In [ ]:
doc.add_story_segment(
    name="The ground it came off, before and after",
    center=(85.5204, 28.289),
    zoom=12.8,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="52%",
    transition_time=1.8,
    layer_overrides=show(detachment, lakes),
    markdown="""**The same 4.5 km square, before the flood and after it.**

![Where it came off, before and after](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-detachment-before-after.png)

Before the flood, the brown outline held dark rock and a tongue of ice. In the
second picture the ice is gone from the part the satellite could see. The ground
under it is grey and brown.

This is the highest and cloudiest box in the map. In the second picture, 38 per
cent of the ground never got a clear look. In the first, only 1 per cent.

The teal outline on the left is one of the two lakes this story ends on.
""",
)

In [ ]:
doc.add_story_segment(
    name="Ten minutes later, the gauge stopped",
    center=(85.34, 28.16),
    zoom=10.6,
    pitch=45,
    bearing=30,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.2,
    layer_overrides=show(reaches, destroyed),
    markdown="""**Ten minutes later, the river gauge at Syafrubesi stopped sending.**

Its last reading was **1.62 m**. The gauge sits twenty kilometres below the
slope. It never sent again.

By **09:15** the Department of Hydrology and Meteorology had confirmed a flood
wave. It warned people down the Trishuli. **679,295 text
messages** went out to phones along the river.

The warning reached everyone below the gauge. The people above it already had
the water.

**On the map:** red dots are what was later marked destroyed
""",
)

In [ ]:
doc.add_story_segment(
    name="The steep gorge",
    center=(85.3694, 28.2587),
    zoom=11.7,
    pitch=45,
    bearing=30,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.2,
    layer_overrides=show(reaches, destroyed, bridges),
    markdown="""**Here the riverbed drops 42.5 m every kilometre, and the valley walls are close
together.**

The river used to be a thin line through forest. It is now bare rock from one
wall to the other. In this 6 by 6 km box, **715 features** are marked destroyed.
That counts buildings, roads and bridges together. Three health posts were lost
too: at Timure, Rasuwagadhi and Syafrubesi.

On ground this steep, the water hits instead of rising.

**On the map:** bridges, dark red where lost
""",
)

In [ ]:
doc.add_story_segment(
    name="The steep gorge, before and after",
    center=(85.3694, 28.2587),
    zoom=11.7,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="52%",
    transition_time=1.8,
    layer_overrides=show(reaches, destroyed, bridges),
    markdown="""**Timure and Rasuwagadhi: the same six kilometre square, before the flood and
after it.**

![Timure and Rasuwagadhi, before and after](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-before-after-red.png)

A thin line of water through forest became bare rock and gravel, from one valley
wall to the other. The village went with the bank it stood on.

Grey means no satellite got a clear look in the days after. Every pass was 70 to
90 per cent cloud. So each picture is built from the clear pixels of several
passes. That is true of every picture after this one too.
""",
)

In [ ]:
doc.add_story_segment(
    name="The middle valley",
    center=(85.3399, 28.1662),
    zoom=11.7,
    pitch=45,
    bearing=20,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.4,
    layer_overrides=show(reaches, extent_hot, destroyed),
    markdown="""**Six kilometres on, the bed is flatter, at 24.3 m per km. The valley is opening
out.**

A box of the same size holds **342** destroyed. That is less than half the gorge.
The ground is gentler here, so more people build on it. Downstream in Nuwakot
alone, **1,267 houses** were fully damaged and
**1,253** partly damaged.

**On the map:** the blue fill is the water line that was observed
""",
)

In [ ]:
doc.add_story_segment(
    name="The middle valley, before and after",
    center=(85.3399, 28.1662),
    zoom=11.7,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="52%",
    transition_time=1.8,
    layer_overrides=show(reaches, extent_hot, destroyed),
    markdown="""**Syabru Besi: the same six kilometre square, before the flood and after it.**

![Syabru Besi, before and after](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-before-after-orange.png)

The river was already wide here. It came out wider still, and pale with new sand.
It took the fields and the edge of the road. It did not take the whole valley
floor.
""",
)

In [ ]:
doc.add_story_segment(
    name="The open valley",
    center=(85.1033, 27.86),
    zoom=11.7,
    pitch=45,
    bearing=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.4,
    layer_overrides=show(reaches, extent_hot, destroyed),
    markdown="""**Past Bidur the bed drops only 2.6 m per km. The river finally has room.**

You can still see the channel, but the valley is wide enough to hold it. The same
box holds **154** destroyed, against 342 and 715 upstream.

**715, then 342, then 154**, in three boxes of the same size down one river.
""",
)

In [ ]:
doc.add_story_segment(
    name="The open valley, before and after",
    center=(85.1033, 27.86),
    zoom=11.7,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="52%",
    transition_time=1.8,
    layer_overrides=show(reaches, extent_hot, destroyed),
    markdown="""**Below Bidur: the same six kilometre square, before the flood and after it.**

![below Bidur, before and after](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-before-after-yellow.png)

Here the change is small: a wider, paler channel and more sand. This is the same
flood that destroyed the gorge. Here the valley was wide enough to hold it.
""",
)

In [ ]:
doc.add_story_segment(
    name="Why the three are different",
    center=(84.8649, 28.0041),
    zoom=9.2,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="58%",
    transition_time=2.2,
    layer_overrides=show(reaches, bridges),
    markdown="""**One thing explains all three: how steeply the riverbed falls.**

![Bed gradient and bridge losses over the same three reaches](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-gradient-vs-damage.png)

A steep bed makes fast water. Fast water hits the bank with everything it is
carrying. Where the bed is flat, the river spreads out instead of cutting in.

**16 of 16** bridges were lost in the
gorge. **6 of 108** on the plain. In its report of
1 September, NDRRMA counted **41 road bridges washed
away** and 4 damaged along the whole river.
""",
)

In [ ]:
doc.add_story_segment(
    name="Where it got to, and when",
    center=(84.58, 27.82),
    zoom=8.4,
    pitch=40,
    bearing=20,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[
        *show(reaches, extent_hot, timed),
        *flowing(0, TIMED[-1][0], seconds=12),
    ],
    markdown="""**The wave took most of a working day to cross the country. People timed it the
whole way.**

Galchhi **10:28** · Malekhu **11:50** · Mugling **13:00** · Devghat **15:20**. At Devghat it peaked about an hour later, at some **5,850 m³/s**.

Those four places are 90, 115, 155 and 189 km down the river from the
slope.

**57.4 million m³** of water passed in all. About
20.0 million of that was on top of what the
river was already carrying.

**On the map:** four violet dots down the yellow reach, upstream to downstream
in the order above, getting darker as the day goes on. The blue line is the water
running down the river through the day, and the clock shows the time
""",
)

In [ ]:
doc.add_story_segment(
    name="About 09:00, Timure",
    **chase(0),
    zoom=12.6,
    pitch=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[*show(reaches, timed), *following(0, 13.7, seconds=10)],
    markdown="""**About 09:00. The water reaches Timure, 14 km down.**

ICIMOD says the Bhote Koshi began rising around 9:00. This story reads that as the water reaching Timure, 23 minutes after the slope gave way.

**On the map:** the blue line is where the water had got to by then.
""",
)

In [ ]:
doc.add_story_segment(
    name="10:28, Galchhi",
    **chase(75.36),
    zoom=12.6,
    pitch=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[*show(reaches, timed), *following(75.36, 90.36, seconds=12)],
    markdown="""**10:28. Galchhi, 90 km down.**

From here on the times are the ones people reported. Between two timed places the blue line is spread evenly by distance. Nobody timed those stretches.
""",
)

In [ ]:
doc.add_story_segment(
    name="11:50, Malekhu",
    **chase(100.03),
    zoom=12.6,
    pitch=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[*show(reaches, timed), *following(100.03, 115.03, seconds=12)],
    markdown="""**11:50. Malekhu, 115 km down.**
""",
)

In [ ]:
doc.add_story_segment(
    name="13:00, Mugling",
    **chase(139.75),
    zoom=12.6,
    pitch=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[*show(reaches, timed), *following(139.75, 154.75, seconds=12)],
    markdown="""**13:00. Mugling, 155 km down.**
""",
)

In [ ]:
doc.add_story_segment(
    name="15:20, Devghat",
    **camera(188.86, looking_back=True),
    zoom=9.4,
    pitch=45,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=[*show(reaches, timed), *flowing(154.75, TIMED[-1][0], seconds=8)],
    markdown="""**15:20. Devghat, 189 km down, 6 h 43 min after the slope gave way.**

It peaked here about an hour later, at some 5,850 m³/s. Looking back up the valley, the blue line runs all the way to the slope.
""",
)

In [ ]:
doc.add_story_segment(
    name="Destroyed upstream, killed downstream",
    center=(84.6, 27.82),
    zoom=8.0,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition_time=2.4,
    layer_overrides=show(reaches, destroyed),
    markdown="""**Nearly everything you have just seen destroyed was in the mountains. But most
of the people who died were on the plain.**

Deaths by district, as the Nepal Police counted them on 5 September:

- **Chitwan 360**
- **Nawalparasi West 230**, **East 222**
- Nuwakot 192 · Rasuwa 160 · Gorkha 73 · Dhading 67 · Tanahun 38

812 of 1,342 deaths are in the three districts
furthest from the mountain. That is where the flood was slowest and broke the
least. Rasuwa, where it tore the valley open, lost 160.
Four days earlier that figure was 27.

Almost nobody lives in the gorge. The plain is where the people are. The flood
did the most damage where the ground was steepest. It killed where the people
were.
""",
)

In [ ]:
doc.add_story_segment(
    name="The people",
    center=(85.0, 27.9),
    zoom=8.6,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition_time=2.2,
    layer_overrides=show(reaches, schools),
    markdown="""**As of 7 September, the Kathmandu Post reported NDRRMA's count as:**

- **1,355 dead**
- **4,996 missing** — 2,340 from Nuwakot, 1,967 from Rasuwa, 589 foreign nationals
- **13,396 rescued**, and **6,672** treated for injuries

For six days after the water, **3,702 households,
14,461 people** in Gosaikunda and Amachhodingmo were
completely cut off. No road, no electricity, no telephone.

**8 schools** were destroyed and 8 damaged,
out of 19 checked. Together they taught
6,901 pupils.

Those last two figures are from 1 September. The district list on the card before
came to 1,342, counted by the Nepal Police two days earlier. Different bodies
count on different days, and [NDRRMA](https://bipadportal.gov.np/) publishes the
current ones.

**On the map:** schools inside the flood extent, red where destroyed
""",
)

In [ ]:
doc.add_story_segment(
    name="What is still up there",
    center=(85.495, 28.312),
    zoom=12.2,
    pitch=45,
    bearing=120,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.6,
    layer_overrides=show(detachment, lakes),
    markdown="""**The debris blocked the river in places. Those blockages are still forming, and
still breaking.**

UNOSAT saw two barrier lakes on images from 28 August,
**20 ha** and
**12 ha**. NDRRMA report that some have since
drained, and that more can form while the banks are loose and the rain keeps
coming.

A pile of debris across a river is not a dam anyone built. Of 73 landslide dams
whose lifetime was known, Costa and Schuster found
**27 per cent broke within a day**, and about half within ten days. Those shares
only describe dams that broke. Many never do.

Pawan Bhattarai of Tribhuvan University, quoted by Al Jazeera on 28 August: "The
danger is not just the lake itself, but the possibility of sudden dam failure."

The valley below has lost its road and most of its bridges.

**On the map:** the teal shapes are the two lakes · brown is where the mountain
broke away
""",
)

In [ ]:
doc.add_story_segment(
    name="The two lakes, before and after",
    center=(85.495, 28.312),
    zoom=12.4,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition_time=1.8,
    layer_overrides=show(detachment, lakes),
    markdown="""**The same two boxes, before the flood and after it.**

![The two barrier lakes, before and after](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-barrier-lakes.png)

The upper lake, 20 ha, sits 160 m below the edge
of the scar. It is held in by what came off the mountain. The lower lake,
12 ha, is 4.6 km north-west. Its valley has gone
from forest to pale sand.

In normal colour, pale ground and water look alike. In near infrared they do not,
because water absorbs it. Inside the upper outline, the water index goes from
-0.01 before the flood to +0.05 after. The share of pixels above zero goes from
39 to 99 per cent. Water would move it in that direction, across about that much
of the outline. But it would usually move it further.

The outlines are UNOSAT's, from CARTOSAT-3 of 28 August.
""",
)

In [ ]:
doc.add_story_segment(
    name="What could still happen",
    center=(85.46, 28.3),
    zoom=11.4,
    pitch=45,
    bearing=120,
    pane_alignment="start",
    panel_width="30rem",
    transition_time=2.2,
    layer_overrides=show(detachment, lakes),
    markdown="""**Both lakes sit above the valley the flood has just been through.**

On 27 August, China's Ministry of Water Resources put one of them at
**1.5 to 2 million m³**. It expected 3 million more within three days.

What happened next is reported three different ways. Rasuwa officials said on
28 August that the lake had started to overflow. The Kathmandu Post reported the
same day that the Army had told the district officer it had broken through. The
Stimson Center, reading Planet images, wrote that both lakes were still growing.

Stimson were more worried about the slope than about the lakes. They warned that
smaller slides could start again if it rained hard in the days after. ICIMOD has
asked for monitoring across the region, because this one started on the Nepali
side, outside the Chinese early-warning area.
""",
)

In [ ]:
doc.add_story_segment(
    name="Radar, through the cloud",
    center=(85.5065, 28.307),
    zoom=12.2,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition_time=1.8,
    layer_overrides=show(detachment, lakes),
    markdown="""**Every camera picture in this map has a hole in it. Radar can see through cloud.**

![Backscatter change, 16 to 28 August](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-radar-change.png)

This is two Sentinel-1 radar passes of the same orbit, twelve days apart, one
subtracted from the other. The angle is the same both times. So the shape of the
land cancels out, and what is left is change on the ground. Smooth surfaces
bounce the radar away and come out dark. Standing water, wet mud and fresh sand
are all smoother than what was there before. So a drop in the radar signal means
the surface got smoother. It does not say what made it smoother.

Across the whole box, **4 per cent** of pixels dropped by more than 3 dB. Inside
UNOSAT's scar, **29 per cent**. Inside the lower lake, **57 per cent**.

The upper lake is the exception, at 23 per cent, and its middle value barely
moves. The camera pictures say that outline filled with something. The radar
shows the same ground barely changing. This map will not decide between them.
""",
)

In [ ]:
doc.add_story_segment(
    name="Four counts of the same thing",
    center=(85.1609, 28.0393),
    zoom=9.8,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="52%",
    transition_time=2.2,
    layer_overrides=show(reaches, graded),
    markdown="""**No count of this flood is final, and the counts do not agree with each other.**

Four organisations counted destroyed buildings. They got
**2,813**, **5,048**, **1,641** and **677**.
They covered different ground, used different methods, and did not mean the same
thing by destroyed. None of them is the number.

![The same ground graded twice, two days apart](https://raw.githubusercontent.com/arjxn-py/nepal-flood-26-aug/main/data/nepal-flood-2026-review/fig-reflight.png)

Copernicus graded one area on 29 August, then looked again on 31 August. Nothing
on the ground changed in between. The second look found
**653 more buildings
destroyed** than the first.

It has since re-graded a second area and found
**105 more**. Both times the number went up, and only
up.

The dead are counted the same way. Between 1 and 5 September, Chitwan's toll
grew by about a tenth, from **321 to 360**. Rasuwa's grew
almost six times, from **27 to 160**. Those two counts
come from different bodies, so the change is worth more than the difference.
But the mountains are the hard place to count.

Early counts are too low. So are early maps. The volunteer record stops at
**km 78**. That is where one mapping project ended and the next
one began. It is not where the flood ended. The water ran on for another 130 km
past the last red dot.
""",
)

In [ ]:
doc.add_story_segment(
    name="Sources",
    center=(84.8649, 28.0041),
    zoom=9.0,
    content_mode="markdown",
    pane_alignment="start",
    panel_width="34%",
    transition_time=2,
    layer_overrides=show(reaches),
    markdown="""**Every number here is either measured from the data in this folder, or quoted
from a named source, with the time it was published.**

**The people and the timeline**

[NDRRMA Situation Report #01](https://ndrrma.gov.np/mediafiles/rasuwa/Rasuwa_Flood_SitRep_Temp_ENG_01_01092026.pdf), 1 September 2026, 09:00.
The cut-off villages, the bridges, the schools, the health posts, the arrival
times and the account of the cause all come from it. It is the last situation
report NDRRMA has published in English.

Those numbers changed daily, and the later ones here are reported rather than
measured. [The Kathmandu Post, 5 September 2026](https://kathmandupost.com/national/2026/09/05/nepal-flood-death-toll-reaches-1-342),
carrying the Nepal Police district-by-district count of 8:00 that morning. [The
Kathmandu Post, 7 September 2026](https://kathmandupost.com/national/2026/09/07/nepal-flood-death-toll-rises-to-1-355-nearly-5-000-unaccounted-for),
carrying NDRRMA's count of 16:20 that day. [BIPAD](https://bipadportal.gov.np/)
carries the current ones.

**The damage mapping**

- [Humanitarian OpenStreetMap Team](https://data.humdata.org/dataset/hot_flood_npl). Flood extent, destroyed features, bridges, schools. ODbL 1.0.
- [Copernicus EMS, EMSR927](https://data.humdata.org/dataset/npl-flood-emsr927). Damage grading. © 2026 European Union.
- [UNOSAT, FL20260826NPL](https://data.humdata.org/dataset/mudflow-rockflow-impact-assessment-in-rasuwa-nuwakot-districts-bagmati-province-nepal-as-o). Flood extent, detachment zone, barrier lakes.

**The barrier lakes**

- Costa, J. E. and Schuster, R. L., 1988, [The formation and failure of natural dams](https://pubs.usgs.gov/publication/70015581), Geological Society of America Bulletin, v. 100, p. 1054-1068. The 73-dam longevity figures, and the warning that they only describe dams that broke.
- [Al Jazeera, 28 August 2026](https://www.aljazeera.com/news/2026/8/28/is-nepal-facing-another-devastating-flood-from-a-new-lake). The Chinese volume estimate, the overflow report, and the quotation.
- [The Kathmandu Post, 28 August 2026](https://kathmandupost.com/national/2026/08/28/barrier-lake-near-china-border-breaches-raising-fresh-flood-threat-in-nepal). The breach report.
- [Stimson Center, 72 hours later](https://www.stimson.org/2026/a-cascading-disaster-on-the-china-nepal-border-what-to-know-about-the-august-2026-rasuwa-flood/). Both lakes growing, and the risk of the slope moving again.
- [ICIMOD](https://www.icimod.org/kyirong-rasuwa-flood-2026-nepal-china-border/). The cause, and the case for cross-border monitoring.

Those five are news reports and published research, not measurements. They are
quoted that way. Where they disagree, the story says so instead of picking one.

The water index is NDWI: (green - near infrared) / (green + near infrared). It
was sampled inside the UNOSAT outlines, on the same Sentinel-2 pictures the
figure draws. `METHODS.md` has the numbers, and what they do not settle.

**The imagery**

Sentinel-2, 10 m, openly licensed. Every pass after the flood is 70-90% cloud.
So each before-and-after is built from the clear pixels of several passes. Ground
that never got a clear look is left as a gap.

The agencies worked from much sharper pictures: Pléiades Neo and WorldView-3 at
0.3 m, plus CARTOSAT-3 and SkySat. Those are commercial and restricted. The
outlines drawn from them can be shared. The images cannot.

The radar is Sentinel-1 RTC gamma0 from the Microsoft Planetary Computer, one
matched pair on relative orbit 85. It is the only picture here with no cloud in
it. It is also the only one that shows a difference rather than a photograph.

**The channel and the reaches**

Measured from terrain tiles for [an earlier map of this
flood](https://github.com/arjxn-py/nepal-flood-26-aug). That map was made in the
week before any of the damage data existed.

The damage mapping on this page was taken from HDX on **8 September 2026**. It
is revised daily, so the counts here are a reading of it on that date, not a
final figure.

## Explore the map yourself

**Everything this page showed you is still in the file, and you can open it
[here](https://notebook.link/@arjxn-py/nffr).** The data it reads, and the
methods behind it, are in
[the repository](https://github.com/arjxn-py/nepal-flood-26-aug).

**Ask a feature where it came from.** Turn on the identify tool in the toolbar
and click any dot, line or patch. Every feature carries its source, the link to
that source, its licence and the date it was fetched.

**Turn layers on and off.** The story only ever showed a few at a time. The
layer panel has all of them, including the ones it never put on screen.
""",
)

In [ ]:
doc